# Occupancy Zero-Aware Test V1

I created this notebook inside `ml/test` to test the zero-aware hurdle idea against the current occupancy target. The first thing I need to verify is whether the target actually contains zero-occupancy rows.


## Setup Note

In this cell I import the shared ML helper and resolve paths so I can run the notebook directly from the `test` folder without breaking imports.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

BASE_DIR = Path.cwd()
SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR / 'code files' / 'ml',
    BASE_DIR / 'code files' / 'ml' / 'test',
    BASE_DIR.parent,
    BASE_DIR.parent / 'ml',
    BASE_DIR.parent.parent,
    BASE_DIR.parent.parent / 'ml',
]
for candidate in SEARCH_DIRS:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from gold_ml_modeling import build_mysql_engine, load_property_mart_frame, prepare_common_features

TEST_DIR = Path.cwd()
TEST_DIR


## Data Load Note

In this cell I load the current Gold property mart and prepare the shared occupancy features before checking the target distribution.


In [ ]:
engine = build_mysql_engine()
model_df = load_property_mart_frame(engine)
prepared_df = prepare_common_features(model_df)
occupancy = prepared_df['target_occupancy_rate'].dropna()
len(occupancy)


## Zero Diagnostic Note

In this cell I measure how many exact zeros and near-zero rows exist in the current occupancy target. If there are no zeros, then a zero-aware hurdle model is not appropriate for this dataset.


In [ ]:
diagnostic_df = pd.DataFrame([
    {'metric': 'occupancy_rows', 'value': int(len(occupancy))},
    {'metric': 'occupancy_zero_count', 'value': int((occupancy == 0).sum())},
    {'metric': 'occupancy_zero_share', 'value': float((occupancy == 0).mean())},
    {'metric': 'occupancy_small_le_0_05_count', 'value': int((occupancy <= 0.05).sum())},
    {'metric': 'occupancy_small_le_0_05_share', 'value': float((occupancy <= 0.05).mean())},
])
diagnostic_df.to_csv(TEST_DIR / 'occupancy_zero_aware_diagnostic_v1.csv', index=False)
diagnostic_df


## Conclusion Note

In this cell I make the conclusion explicit. The current occupancy target contains no zero rows, so the requested zero-aware two-stage classifier-plus-regressor strategy is not applicable to the current training target as it stands.


In [ ]:
zero_count = int((occupancy == 0).sum())
if zero_count == 0:
    conclusion = pd.Series({
        'status': 'not_applicable_on_current_target',
        'reason': 'the current occupancy target has zero exact zero-occupancy rows',
        'recommended_next_step': 'if we want a two-stage occupancy model, we should redefine the target around true period-level occupancy or a low-demand threshold instead of exact zeros',
    })
else:
    conclusion = pd.Series({
        'status': 'applicable',
        'reason': 'the target contains exact zeros',
    })
conclusion
